## Step 3 of the migration: restore data in the NEW workspace

Run by the bundle job **Restore FEMA hackathon data** after `databricks bundle deploy -t new_env`. Every step is safe to re-run (it skips work already done).

1. Copies the submission packet files into the new `hackathon_submissions` volume
2. Recreates the golden eval dataset (116 cases) as an MLflow evaluation dataset
3. Recreates the Genie space *FEMA Public Assistance Funding* on your warehouse
4. Re-logs past MLflow evaluation runs (params, metrics, tags; original run ID kept as tag `original_run_id`)

The job's second task then runs **Load FEMA Funded Projects** to reload `funded_projects_details` from OpenFEMA.

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "default")
dbutils.widgets.text("volume_name", "hackathon_submissions")
dbutils.widgets.text("warehouse_id", "")
dbutils.widgets.text("experiment_name", "")
dbutils.widgets.text("bundle_files_path", "")

import hashlib, json, os, shutil
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/{dbutils.widgets.get('volume_name')}"
WAREHOUSE_ID = dbutils.widgets.get("warehouse_id")
EXPERIMENT_NAME = dbutils.widgets.get("experiment_name")
FILES = dbutils.widgets.get("bundle_files_path")
if not FILES.startswith("/Workspace"):
    FILES = "/Workspace" + FILES
DATA = f"{FILES}/migration_data"
assert os.path.isdir(DATA), f"Snapshot folder not found: {DATA}"
assert WAREHOUSE_ID, "warehouse_id is required"
print("Restoring into", CATALOG, SCHEMA, "| volume", VOLUME, "| experiment", EXPERIMENT_NAME)

def load(name):
    with open(f"{DATA}/{name}") as f:
        return json.load(f)

def sha(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()

In [0]:
src_root = f"{DATA}/volume_files"
copied = skipped = 0
for root, _, files in os.walk(src_root):
    for name in files:
        src = os.path.join(root, name)
        dst = os.path.join(VOLUME, os.path.relpath(src, src_root))
        if os.path.exists(dst) and sha(dst) == sha(src):
            skipped += 1
            continue
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copyfile(src, dst)
        assert sha(dst) == sha(src), f"Copy check failed: {dst}"
        copied += 1
print(f"Volume files copied: {copied}, already present: {skipped}")

In [0]:
import mlflow
import mlflow.genai.datasets as gd

mlflow.set_tracking_uri("databricks")
exp = mlflow.set_experiment(EXPERIMENT_NAME)
GOLDEN = f"{CATALOG}.{SCHEMA}.fema_oversight_golden_v1"
records = load("golden_dataset.json")

if spark.catalog.tableExists(GOLDEN) and spark.table(GOLDEN).count() >= len(records):
    print(f"{GOLDEN} already has {spark.table(GOLDEN).count()} rows - skipped")
else:
    try:
        ds = gd.get_dataset(uc_table_name=GOLDEN)
    except Exception:
        ds = gd.create_dataset(uc_table_name=GOLDEN, experiment_id=exp.experiment_id)
    ds.merge_records([{"inputs": r["inputs"], "expectations": r["expectations"], "tags": r.get("tags") or {}}
                      for r in records])
    print(f"{GOLDEN}: {spark.table(GOLDEN).count()} rows (expected {len(records)})")

In [0]:
g = load("genie_space.json")
serialized = g["serialized_space"]
if (CATALOG, SCHEMA) != ("workspace", "default"):
    serialized = serialized.replace("workspace.default.", f"{CATALOG}.{SCHEMA}.")

existing = w.api_client.do("GET", "/api/2.0/genie/spaces").get("spaces", [])
match = [s for s in existing if s.get("title") == g["title"]]
if match:
    print("Genie space already exists:", match[0]["space_id"], "- skipped")
else:
    created = w.api_client.do("POST", "/api/2.0/genie/spaces", body={
        "title": g["title"], "description": g.get("description") or "",
        "warehouse_id": WAREHOUSE_ID, "serialized_space": serialized})
    print("Genie space created:", created.get("space_id"), g["title"])

In [0]:
runs = load("mlflow_runs.json")
prior = mlflow.search_runs(experiment_ids=[exp.experiment_id], output_format="list")
done = {p.data.tags.get("original_run_id") for p in prior} - {None}
logged = 0
for r in runs:
    if r["run_id"] in done:
        continue
    with mlflow.start_run(experiment_id=exp.experiment_id, run_name=r.get("run_name")):
        params = {k: str(v)[:6000] for k, v in (r.get("params") or {}).items()}
        for i in range(0, len(params), 100):
            mlflow.log_params(dict(list(params.items())[i:i + 100]))
        if r.get("metrics"):
            mlflow.log_metrics(r["metrics"])
        tags = {k: str(v)[:5000] for k, v in (r.get("tags") or {}).items() if k != "mlflow.runName"}
        tags.update({"original_run_id": r["run_id"], "migrated_from": "old hackathon workspace",
                     "original_start_time": str(r.get("start_time"))})
        mlflow.set_tags(tags)
    logged += 1
print(f"Runs re-logged: {logged}, already present: {len(runs) - logged}")